# Kernel ZZ en pytket ejecutado desde Guppy

Este cuaderno sigue el flujo de `Parte Cuantica v1.ipynb`:

1. carga las observaciones escaladas;
2. construye y muestra el feature map \(U(x)\) en **pytket**;
3. elige dos filas y muestra \(U(x_j)^\dagger U(x_i)\);
4. carga ese circuito pytket en Guppy;
5. solo cuando `RUN_KERNEL = True`, lo ejecuta en Selene local o Nexus.

La definición del circuito permanece en pytket. Guppy se usa como envoltura de
compilación, medición y ejecución.

In [26]:
import importlib
import uuid
from pathlib import Path

import numpy as np
import pandas as pd
import qnexus as qnx
from IPython.display import display
from pytket import Circuit
from pytket.circuit.display import render_circuit_jupyter
from pytket.passes import RemoveBarriers

# Recarga el helper para no conservar una firma vieja en memoria.
import funciones_nexus as nexus_helpers
importlib.reload(nexus_helpers)

from funciones_nexus import (
    compilar_y_subir_hugr,
    conectar_nexus,
    construir_tabla_resultados,
    descargar_resultados_job,
    ejecutar_local,
    enviar_job_selene,
    guardar_ejecucion_csv,
    nuevo_run_id_local,
)

## 1. Datos escalados

Se usan las mismas variables escaladas que consume `Parte Cuantica v1`.
`kernel_train_df.iloc[k]` representa la fila `k` dentro de la partición train.

In [27]:
KERNEL_DATA_PATH = Path("data/raw/df_escalado.csv")

kernel_df = pd.read_csv(KERNEL_DATA_PATH, sep=";")
kernel_feature_columns = [
    column
    for column in kernel_df.columns
    if column not in {"Potability", "_PartInd_"}
]

kernel_train_df = (
    kernel_df.loc[kernel_df["_PartInd_"] == 0, kernel_feature_columns]
    .reset_index(drop=True)
)
kernel_test_df = (
    kernel_df.loc[kernel_df["_PartInd_"] == 1, kernel_feature_columns]
    .reset_index(drop=True)
)

print("Features:", kernel_feature_columns)
print("Train:", kernel_train_df.shape)
print("Test:", kernel_test_df.shape)
display(kernel_train_df.head())

Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Train: (2620, 9)
Test: (656, 9)


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
0,0.505288,0.371295,0.048040,-0.791809,2.052308,0.925183,0.565242,-0.365393,-0.177457
1,-0.709433,0.935084,-1.016080,0.850917,1.617777,-1.501693,-2.169448,-1.581404,-0.486502
2,0.929048,0.980710,0.808192,-0.585714,-0.019216,-0.818137,0.085404,-0.351573,0.547202
3,-0.227728,0.671248,-0.001853,0.030738,-0.019216,-0.677955,0.774861,0.501623,-0.091554
4,-0.390242,0.176673,-0.548156,-0.571875,-0.196177,0.665479,0.686438,-1.263826,-0.544746


## 2. Feature map ZZ en pytket

Primero se inspecciona únicamente \(U(x)\). Las barreras ayudan a visualizar
las etapas de embedding individual y correlaciones ZZ.

In [28]:
def zz_feature_map(x):
    '''Construye U(x) enteramente en pytket.'''
    x = np.asarray(x, dtype=float)
    n_qubits = len(x)

    circuit = Circuit(n_qubits, name="ZZ Feature Map")
    qubits = list(range(n_qubits))

    # Embedding individual
    for i in range(n_qubits):
        circuit.H(i)

    # pytket expresa los angulos en medias vueltas.
    for i in range(n_qubits):
        circuit.Rz(2 * x[i] / np.pi, i)

    circuit.add_barrier(qubits)

    # Correlaciones ZZ
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            angle = 2 * (np.pi - x[i]) * (np.pi - x[j])
            circuit.CX(i, j)
            circuit.Rz(angle / np.pi, j)
            circuit.CX(i, j)
            circuit.add_barrier(qubits)

    return circuit

In [29]:
# Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots.
PREVIEW_ROW = 0

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_preview = zz_feature_map(preview_x)

print(f"Feature map de train[{PREVIEW_ROW}]")
print("Qubits:", feature_map_preview.n_qubits)
print("Puertas:", feature_map_preview.n_gates)
render_circuit_jupyter(feature_map_preview)

Feature map de train[0]
Qubits: 9
Puertas: 163


## 3. Circuito del kernel

Para estimar una entrada se construye

\[
K(x_i,x_j)=\left|\langle 0|U(x_j)^\dagger U(x_i)|0\rangle\right|^2.
\]

Todavía no se ejecuta nada: primero se seleccionan las filas y se inspecciona
el circuito pytket resultante.

In [30]:
def kernel_circuit_zz(x_i, x_j, remove_barriers=False):
    '''Construye U(x_j)^dagger U(x_i) como un pytket.Circuit.'''
    circuit_xi = zz_feature_map(x_i)
    circuit_xj_adjoint = zz_feature_map(x_j).dagger()

    kernel = circuit_xi.copy()
    kernel.append(circuit_xj_adjoint)

    if remove_barriers:
        RemoveBarriers().apply(kernel)

    return kernel

In [31]:
# Estas son las dos filas que se enviaran al simulador cuando lo habilites.
KERNEL_ROW_I = 0
KERNEL_ROW_J = 1

if min(KERNEL_ROW_I, KERNEL_ROW_J) < 0:
    raise IndexError("Los indices no pueden ser negativos.")
if max(KERNEL_ROW_I, KERNEL_ROW_J) >= len(kernel_train_df):
    raise IndexError("KERNEL_ROW_I o KERNEL_ROW_J queda fuera de train.")

kernel_x_i = kernel_train_df.iloc[KERNEL_ROW_I].to_numpy(dtype=float)
kernel_x_j = kernel_train_df.iloc[KERNEL_ROW_J].to_numpy(dtype=float)

# Con barreras para la inspeccion visual.
kernel_preview_circuit = kernel_circuit_zz(
    kernel_x_i,
    kernel_x_j,
    remove_barriers=False,
)

print(f"Kernel seleccionado: train[{KERNEL_ROW_I}] vs train[{KERNEL_ROW_J}]")
print("Qubits:", kernel_preview_circuit.n_qubits)
print("Puertas:", kernel_preview_circuit.n_gates)
print("Profundidad:", kernel_preview_circuit.depth())

Kernel seleccionado: train[0] vs train[1]
Qubits: 9
Puertas: 326
Profundidad: 220


In [32]:
# Inspecciona U(x_j)^dagger U(x_i) antes de cargarlo en Guppy.
render_circuit_jupyter(kernel_preview_circuit)

## 4. Puente pytket → Guppy

Las barreras se eliminan únicamente de la copia ejecutable. Las puertas y los
ángulos siguen siendo los definidos por pytket.

In [33]:
from guppylang import guppy
from guppylang.std.builtins import array, comptime, result
from guppylang.std.quantum import measure_array, qubit


kernel_tket_circuit = kernel_circuit_zz(
    kernel_x_i,
    kernel_x_j,
    remove_barriers=True,
)
N_QUBITS = kernel_tket_circuit.n_qubits

# Este es el unico puente: no se reescriben las puertas en Guppy.
kernel_zz = guppy.load_pytket(
    "kernel_zz",
    kernel_tket_circuit,
    use_arrays=True,
)


@guppy
def ejecutar_zz_kernel() -> None:
    qs = array(qubit() for _ in range(comptime(N_QUBITS)))
    kernel_zz(qs)
    measurements = measure_array(qs)
    result("kernel_measurement", measurements)


ejecutar_zz_kernel.check()
print("Programa Guppy comprobado; aun no se ha ejecutado.")

Programa Guppy comprobado; aun no se ha ejecutado.


In [34]:
def resumen_kernel_desde_resultado(execution_result, n_qubits):
    """
    Extrae P(00...0) tanto de resultados Guppy como de BackendResult pytket.
    """
    zero_state = "0" * n_qubits

    if hasattr(execution_result, "register_counts"):
        # Guppy/Selene: conteos agrupados por etiqueta de result().
        register_counts = execution_result.register_counts()
        kernel_counts = register_counts["kernel_measurement"]
        shots = int(sum(kernel_counts.values()))
        zero_count = int(kernel_counts.get(zero_state, 0))

    elif hasattr(execution_result, "get_counts"):
        # Circuitos pytket en H1/H2: Counter con outcomes de bits.
        kernel_counts = execution_result.get_counts()
        shots = int(sum(kernel_counts.values()))
        zero_count = 0

        for outcome, count in kernel_counts.items():
            try:
                is_zero = all(int(bit) == 0 for bit in outcome)
            except TypeError:
                is_zero = str(outcome).replace(" ", "") in {
                    zero_state,
                    f"({','.join('0' for _ in range(n_qubits))})",
                }
            if is_zero:
                zero_count += int(count)

    else:
        raise TypeError(
            "Tipo de resultado no soportado: se esperaba un resultado "
            "Guppy con register_counts() o BackendResult con get_counts()."
        )

    kernel_rate = zero_count / shots if shots else 0.0
    return {
        "zero_state": zero_state,
        "zero_count": zero_count,
        "shots": shots,
        "kernel_rate": kernel_rate,
    }


def guardar_resumen_kernel_csv(
    summary,
    row_i,
    row_j,
    source,
    run_id=None,
    job_id=None,
    job_name=None,
    directory="data/runs",
):
    """Guarda una sola fila por ejecucion del kernel."""
    if run_id is None:
        run_id = str(job_id) if job_id is not None else f"local-{uuid.uuid4().hex[:12]}"

    row = {
        "run_id": str(run_id),
        "source": source,
        "job_id": "" if job_id is None else str(job_id),
        "job_name": "" if job_name is None else str(job_name),
        "row_i": int(row_i),
        "row_j": int(row_j),
        "n_qubits": len(summary["zero_state"]),
        "zero_state": summary["zero_state"],
        "zero_count": summary["zero_count"],
        "shots": summary["shots"],
        "kernel_rate": summary["kernel_rate"],
    }

    output_dir = Path(directory)
    output_dir.mkdir(parents=True, exist_ok=True)
    safe_run_id = str(run_id).replace("/", "_").replace("\\", "_")
    output_path = output_dir / f"kernel_run_{safe_run_id}.csv"
    pd.DataFrame([row]).to_csv(output_path, index=False)
    return output_path


def tabla_resumen_kernel(summary, row_i, row_j, source):
    """Crea la vista compacta de una sola fila."""
    return pd.DataFrame([{
        "source": source,
        "row_i": int(row_i),
        "row_j": int(row_j),
        **summary,
    }])

## 5. Ejecución controlada

Revisa arriba las filas, el circuito y los recursos. Cuando estés seguro,
cambia `RUN_KERNEL` a `True`.

- `EXECUTION_TARGET = "local"` usa Selene local y no inicia sesión en Nexus.
- `EXECUTION_TARGET = "nexus_selene"` compila, sube y envía un job remoto.

In [68]:
# Interruptor de seguridad: esta celda no consume shots mientras sea False.
RUN_KERNEL = True
EXECUTION_TARGET = "nexus_selene"  # "local" o "nexus_selene"
n_shots = 1000
seed = 42
SAVE_LOCAL_CSV = True
PROJECT_NAME = "guppy-kernel-encoding"

local_result = None
local_counts = None
local_kernel_summary = None
sim_job_ref = None
sim_result = None
sim_counts = None
sim_result_ids = None

if not RUN_KERNEL:
    print("Ejecucion desactivada.")
    print(
        "Revisa el circuito y luego cambia RUN_KERNEL = True "
        f"para ejecutar train[{KERNEL_ROW_I}] vs train[{KERNEL_ROW_J}]."
    )

elif EXECUTION_TARGET == "local":
    local_result, local_counts = ejecutar_local(
        ejecutar_zz_kernel,
        n_qubits=N_QUBITS,
        n_shots=n_shots,
        seed=seed,
        simulator="statevector",
    )
    local_kernel_summary = resumen_kernel_desde_resultado(
        local_result,
        N_QUBITS,
    )

    print("Simulacion local finalizada.")
    display(tabla_resumen_kernel(
        local_kernel_summary,
        KERNEL_ROW_I,
        KERNEL_ROW_J,
        source="local_statevector",
    ))

    if SAVE_LOCAL_CSV:
        local_csv = guardar_resumen_kernel_csv(
            summary=local_kernel_summary,
            row_i=KERNEL_ROW_I,
            row_j=KERNEL_ROW_J,
            source="local_statevector",
        )
        print("Resumen del kernel guardado en:", local_csv)

elif EXECUTION_TARGET == "nexus_selene":
    project = conectar_nexus(PROJECT_NAME)
    suffix = uuid.uuid4().hex[:8]

    hugr_binary, ref_hugr = compilar_y_subir_hugr(
        ejecutar_zz_kernel,
        f"zz-kernel-{KERNEL_ROW_I}-{KERNEL_ROW_J}-{suffix}",
    )
    sim_job_ref = enviar_job_selene(
        ref_hugr,
        n_qubits=N_QUBITS,
        n_shots=n_shots,
        nombre=f"zz-kernel-selene-{KERNEL_ROW_I}-{KERNEL_ROW_J}-{suffix}",
    )
    submit_status = qnx.jobs.status(sim_job_ref)

    print("Job enviado a Selene/Nexus.")
    print("Job ID:", sim_job_ref.id)
    print("Estado inicial:", submit_status.status)
    print("Usa la siguiente celda para consultar y guardar el resultado compacto.")

else:
    raise ValueError('EXECUTION_TARGET debe ser "local" o "nexus_selene"')

Already logged in. Tokens are valid.
Job enviado a Selene/Nexus.
Job ID: 91da759e-316c-483e-bfb0-f6bdc817a279
Estado inicial: JobStatusEnum.SUBMITTED
Usa la siguiente celda para consultar y guardar el resultado compacto.


In [74]:
# Solo aplica despues de enviar un job con EXECUTION_TARGET="nexus_selene".
# Reejecuta esta celda para consultar el avance sin enviar otro job.
if sim_job_ref is None:
    print("No hay un job remoto nuevo que consultar.")

else:
    status = qnx.jobs.status(sim_job_ref)
    print("Estado:", status.status)
    print("Mensaje:", status.message)

    if "COMPLETED" in str(status.status):
        result_refs, downloaded, counts_list, result_ids = (
            descargar_resultados_job(sim_job_ref)
        )
        sim_result = downloaded[0]
        sim_counts = counts_list[0] if len(counts_list) == 1 else counts_list
        sim_result_ids = result_ids

        remote_kernel_summary = resumen_kernel_desde_resultado(
            sim_result,
            N_QUBITS,
        )
        display(tabla_resumen_kernel(
            remote_kernel_summary,
            KERNEL_ROW_I,
            KERNEL_ROW_J,
            source="nexus_selene_statevector",
        ))

        remote_csv = guardar_resumen_kernel_csv(
            summary=remote_kernel_summary,
            row_i=KERNEL_ROW_I,
            row_j=KERNEL_ROW_J,
            source="nexus_selene_statevector",
            job_id=sim_job_ref.id,
            job_name=getattr(sim_job_ref.annotations, "name", None),
        )
        print("Resumen del kernel guardado en:", remote_csv)

    elif "ERROR" in str(status.status) or "CANCELLED" in str(status.status):
        raise RuntimeError(f"El job termino sin exito: {status}")

    else:
        print("El job sigue en cola o ejecucion. Consulta nuevamente mas tarde.")

Estado: JobStatusEnum.COMPLETED
Mensaje: The job is completed.


RuntimeError: El job seleccionado aun no esta COMPLETED y no tiene resultados finales.

## 6. Construcción controlada de la matriz kernel

Esta sección se usa después de validar visualmente un par. Para las filas
seleccionadas en `MATRIX_ROWS`, ejecuta solamente el triángulo superior y
refleja los valores por simetría:

\[
K_{ij}=K_{ji}.
\]

Cada entrada sigue siendo un circuito pytket cargado mediante
`guppy.load_pytket`. El CSV de la matriz guarda una fila compacta por circuito
ejecutado: índices, conteo de \(00\ldots0\), shots y tasa.

El costo crece cuadráticamente. Con \(m\) filas y sin ejecutar la diagonal se
requieren \(m(m-1)/2\) circuitos.

In [35]:
from tqdm.auto import tqdm


def crear_programa_kernel_guppy(x_i, x_j):
    '''Crea el programa Guppy ejecutable para un par, conservando pytket.'''
    pair_circuit = kernel_circuit_zz(
        x_i,
        x_j,
        remove_barriers=True,
    )
    pair_n_qubits = pair_circuit.n_qubits

    pair_kernel = guppy.load_pytket(
        "pair_kernel",
        pair_circuit,
        use_arrays=True,
    )

    @guppy
    def pair_program() -> None:
        qs = array(qubit() for _ in range(comptime(pair_n_qubits)))
        pair_kernel(qs)
        measurements = measure_array(qs)
        result("kernel_measurement", measurements)

    pair_program.check()
    return pair_program, pair_circuit


def ejecutar_kernel_guppy(
    x_i,
    x_j,
    n_shots=1000,
    seed=42,
    simulator="statevector",
):
    '''Ejecuta un único K(i,j) en Guppy/Selene local.'''
    pair_program, pair_circuit = crear_programa_kernel_guppy(x_i, x_j)
    pair_result, pair_counts = ejecutar_local(
        pair_program,
        n_qubits=pair_circuit.n_qubits,
        n_shots=n_shots,
        seed=seed,
        simulator=simulator,
    )
    summary = resumen_kernel_desde_resultado(
        pair_result,
        pair_circuit.n_qubits,
    )

    return {
        "kernel": summary["kernel_rate"],
        "summary": summary,
        "counts": pair_counts,
        "program": pair_program,
        "pytket_circuit": pair_circuit,
        "raw_result": pair_result,
    }

In [36]:
def construir_matriz_kernel_guppy(
    X,
    row_labels=None,
    n_shots=1000,
    seed=42,
    simulator="statevector",
    ejecutar_diagonal=False,
):
    '''
    Construye una matriz simétrica ejecutando un programa Guppy por par.

    La diagonal puede fijarse en 1 sin ejecutar circuitos, ya que
    K(x_i, x_i)=1 idealmente.
    '''
    if isinstance(X, pd.DataFrame):
        X_values = X.to_numpy(dtype=float)
    else:
        X_values = np.asarray(X, dtype=float)

    if X_values.ndim != 2:
        raise ValueError("X debe tener forma (n_filas, n_features).")

    n_samples = X_values.shape[0]
    if row_labels is None:
        row_labels = list(range(n_samples))
    else:
        row_labels = list(row_labels)

    if len(row_labels) != n_samples:
        raise ValueError("row_labels debe tener una etiqueta por fila de X.")

    matrix = np.zeros((n_samples, n_samples), dtype=float)
    run_rows = []

    if ejecutar_diagonal:
        total_circuits = n_samples * (n_samples + 1) // 2
    else:
        np.fill_diagonal(matrix, 1.0)
        total_circuits = n_samples * (n_samples - 1) // 2

    with tqdm(
        total=total_circuits,
        desc="Matriz kernel con Guppy",
        unit="circuito",
    ) as progress:
        for i in range(n_samples):
            start_j = i if ejecutar_diagonal else i + 1

            for j in range(start_j, n_samples):
                pair_seed = seed + i * n_samples + j
                pair = ejecutar_kernel_guppy(
                    X_values[i],
                    X_values[j],
                    n_shots=n_shots,
                    seed=pair_seed,
                    simulator=simulator,
                )

                value = pair["kernel"]
                matrix[i, j] = value
                matrix[j, i] = value

                run_rows.append({
                    "matrix_i": i,
                    "matrix_j": j,
                    "row_i": row_labels[i],
                    "row_j": row_labels[j],
                    "n_qubits": X_values.shape[1],
                    "zero_state": pair["summary"]["zero_state"],
                    "zero_count": pair["summary"]["zero_count"],
                    "shots": pair["summary"]["shots"],
                    "kernel_rate": value,
                    "seed": pair_seed,
                })

                progress.set_postfix({
                    "rows": f"{row_labels[i]},{row_labels[j]}",
                    "Kij": f"{value:.4f}",
                })
                progress.update(1)

    return {
        "kernel_matrix": matrix,
        "run_summary": pd.DataFrame(run_rows),
        "row_labels": row_labels,
        "n_circuits": total_circuits,
        "n_shots_per_circuit": n_shots,
    }


def guardar_matriz_kernel_run(
    matrix_result,
    source="local_statevector",
    run_id=None,
    job_id=None,
):
    """Guarda un CSV compacto con una fila por circuito ejecutado."""
    if run_id is None:
        run_id = f"matrix-local-{uuid.uuid4().hex[:12]}"

    run_df = matrix_result["run_summary"].copy()
    run_df.insert(0, "job_id", "" if job_id is None else str(job_id))
    run_df.insert(0, "source", source)
    run_df.insert(0, "run_id", str(run_id))

    output_dir = Path("data/runs")
    output_dir.mkdir(parents=True, exist_ok=True)
    safe_run_id = str(run_id).replace("/", "_").replace("\\\\", "_")
    output_path = output_dir / f"kernel_matrix_run_{safe_run_id}.csv"
    run_df.to_csv(output_path, index=False)
    return output_path

In [37]:
# Filas de train que formarán la matriz.
MATRIX_ROWS = [0, 1, 2, 3]

# Cambia solamente esta variable para escoger el backend.
MATRIX_BACKEND_OPTIONS = [
    "local_selene_statevector",
    "nexus_selene_statevector",
    "H1-1LE",
    "H1-Emulator",
    "H2-1LE",
    "H2-Emulator",
    "Helios-1E-lite",
]
MATRIX_BACKEND = "H1-1LE"

RUN_MATRIX = True
MATRIX_SHOTS = 1000
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True
SAVE_MATRIX_RUN = True
MATRIX_PROJECT_NAME = "ZZ-FM"

if MATRIX_BACKEND not in MATRIX_BACKEND_OPTIONS:
    raise ValueError(
        f"Backend desconocido: {MATRIX_BACKEND}. "
        f"Opciones: {MATRIX_BACKEND_OPTIONS}"
    )

MATRIX_EXECUTION_TARGET = (
    "local" if MATRIX_BACKEND == "local_selene_statevector" else "nexus"
)
MATRIX_NEXUS_TARGET = (
    "selene_statevector"
    if MATRIX_BACKEND == "nexus_selene_statevector"
    else MATRIX_BACKEND
)

# Estado de los jobs. No vuelvas a ejecutar esta celda tras un submit:
# utiliza matrix-poll-nexus para refrescar compilación/ejecución.
matrix_result = None
matrix_compile_job_ref = None
matrix_job_ref = None
matrix_pair_metadata = []
matrix_hugr_refs = []
matrix_circuit_refs = []
matrix_compiled_refs = []
matrix_backend_config = None
matrix_program_format = None

if not MATRIX_ROWS:
    raise ValueError("MATRIX_ROWS no puede estar vacío.")
if min(MATRIX_ROWS) < 0 or max(MATRIX_ROWS) >= len(kernel_train_df):
    raise IndexError("Algún índice de MATRIX_ROWS queda fuera de train.")
if len(set(MATRIX_ROWS)) != len(MATRIX_ROWS):
    raise ValueError("MATRIX_ROWS no debe contener índices repetidos.")

matrix_input = kernel_train_df.iloc[MATRIX_ROWS]
matrix_values = matrix_input.to_numpy(dtype=float)
n_matrix_rows = len(MATRIX_ROWS)
planned_circuits = (
    n_matrix_rows * (n_matrix_rows + 1) // 2
    if MATRIX_EXECUTE_DIAGONAL
    else n_matrix_rows * (n_matrix_rows - 1) // 2
)

if not RUN_MATRIX:
    print("Construcción de matriz desactivada.")
    print("Filas seleccionadas:", MATRIX_ROWS)
    print("Backend seleccionado:", MATRIX_BACKEND)
    print("Opciones:", MATRIX_BACKEND_OPTIONS)
    print("Circuitos que se ejecutarían:", planned_circuits)
    print("Cambia RUN_MATRIX = True cuando quieras iniciar.")

elif MATRIX_EXECUTION_TARGET == "local":
    matrix_result = construir_matriz_kernel_guppy(
        X=matrix_input,
        row_labels=MATRIX_ROWS,
        n_shots=MATRIX_SHOTS,
        seed=MATRIX_SEED,
        simulator="statevector",
        ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
    )
    K_selected = pd.DataFrame(
        matrix_result["kernel_matrix"],
        index=MATRIX_ROWS,
        columns=MATRIX_ROWS,
    )
    print("Matriz kernel local:")
    display(K_selected)
    print("Resumen compacto de circuitos ejecutados:")
    display(matrix_result["run_summary"])

    if SAVE_MATRIX_RUN:
        matrix_csv = guardar_matriz_kernel_run(
            matrix_result,
            source="local_statevector",
        )
        print("Run de matriz guardado en:", matrix_csv)

elif MATRIX_EXECUTION_TARGET == "nexus":
    matrix_project = conectar_nexus(MATRIX_PROJECT_NAME)
    matrix_suffix = uuid.uuid4().hex[:8]

    if MATRIX_NEXUS_TARGET == "selene_statevector":
        matrix_program_format = "hugr"
        matrix_backend_config = qnx.models.SeleneConfig(
            n_qubits=matrix_input.shape[1],
            simulator=qnx.models.StatevectorSimulator(),
        )
    elif MATRIX_NEXUS_TARGET.startswith("Helios-"):
        matrix_program_format = "hugr"
        matrix_backend_config = qnx.models.HeliosConfig(
            system_name=MATRIX_NEXUS_TARGET,
        )
    else:
        # H1/H2 no aceptan HUGR directo. Se usa el Circuit de pytket.
        matrix_program_format = "pytket_circuit"
        matrix_backend_config = qnx.models.QuantinuumConfig(
            device_name=MATRIX_NEXUS_TARGET,
        )

    with tqdm(
        total=planned_circuits,
        desc=f"Preparando {matrix_program_format}",
        unit="circuito",
    ) as progress:
        for i in range(n_matrix_rows):
            start_j = i if MATRIX_EXECUTE_DIAGONAL else i + 1

            for j in range(start_j, n_matrix_rows):
                pair_name = (
                    f"zz-matrix-{MATRIX_ROWS[i]}-{MATRIX_ROWS[j]}-"
                    f"{matrix_suffix}"
                )
                metadata = {
                    "matrix_i": i,
                    "matrix_j": j,
                    "row_i": MATRIX_ROWS[i],
                    "row_j": MATRIX_ROWS[j],
                    "seed": MATRIX_SEED + i * n_matrix_rows + j,
                }

                if matrix_program_format == "hugr":
                    pair_program, _ = crear_programa_kernel_guppy(
                        matrix_values[i],
                        matrix_values[j],
                    )
                    _, pair_ref = compilar_y_subir_hugr(
                        pair_program,
                        pair_name,
                    )
                    matrix_hugr_refs.append(pair_ref)

                else:
                    pair_circuit = kernel_circuit_zz(
                        matrix_values[i],
                        matrix_values[j],
                        remove_barriers=True,
                    )
                    pair_circuit.measure_all()
                    pair_ref = qnx.circuits.upload(
                        circuit=pair_circuit,
                        name=pair_name,
                    )
                    matrix_circuit_refs.append(pair_ref)

                matrix_pair_metadata.append(metadata)
                progress.update(1)

    if matrix_program_format == "hugr":
        matrix_job_ref = qnx.start_execute_job(
            programs=matrix_hugr_refs,
            n_shots=[MATRIX_SHOTS] * len(matrix_hugr_refs),
            backend_config=matrix_backend_config,
            name=f"zz-kernel-matrix-{MATRIX_NEXUS_TARGET}-{matrix_suffix}",
        )
        print("Job de ejecución HUGR enviado.")
        print("Execute Job ID:", matrix_job_ref.id)

    else:
        matrix_compile_job_ref = qnx.start_compile_job(
            programs=matrix_circuit_refs,
            backend_config=matrix_backend_config,
            optimisation_level=2,
            skip_intermediate_circuits=True,
            name=f"compile-zz-matrix-{MATRIX_NEXUS_TARGET}-{matrix_suffix}",
        )
        print("Circuitos pytket subidos; compile job enviado.")
        print("Compile Job ID:", matrix_compile_job_ref.id)

    print("Backend:", MATRIX_BACKEND)
    print("Formato:", matrix_program_format)
    print("Programas:", len(matrix_pair_metadata))
    print("Ejecuta matrix-poll-nexus para continuar sin reenviar.")

else:
    raise ValueError("Destino de ejecución no reconocido.")

Already logged in. Tokens are valid.


Preparando pytket_circuit:   0%|          | 0/10 [00:00<?, ?circuito/s]

Circuitos pytket subidos; compile job enviado.
Compile Job ID: c8ef3f5b-8ff6-42c7-9e42-bfa6e24d7e82
Backend: H1-1LE
Formato: pytket_circuit
Programas: 10
Ejecuta matrix-poll-nexus para continuar sin reenviar.


In [43]:
# Refresca primero la compilación (H1/H2) y luego la ejecución.
# Reejecuta solamente esta celda; no reejecutes matrix-control tras el submit.

terminal_errors = {"ERROR", "CANCELLED", "TERMINATED", "DEPLETED"}

if matrix_compile_job_ref is not None and matrix_job_ref is None:
    compile_status = qnx.jobs.status(matrix_compile_job_ref)
    print("Compile status:", compile_status.status)
    print("Compile message:", compile_status.message)

    if compile_status.status == qnx.jobs.JobStatusEnum.COMPLETED:
        fresh_compile_ref = qnx.jobs.get(id=matrix_compile_job_ref.id)
        compile_results = list(qnx.jobs.results(fresh_compile_ref))
        matrix_compiled_refs = [item.get_output() for item in compile_results]

        if len(matrix_compiled_refs) != len(matrix_pair_metadata):
            raise RuntimeError(
                "La cantidad de circuitos compilados no coincide con los pares."
            )

        matrix_job_ref = qnx.start_execute_job(
            programs=matrix_compiled_refs,
            n_shots=[MATRIX_SHOTS] * len(matrix_compiled_refs),
            backend_config=matrix_backend_config,
            name=(
                f"execute-zz-matrix-{MATRIX_NEXUS_TARGET}-"
                f"{uuid.uuid4().hex[:8]}"
            ),
        )
        print("Compilación terminada; execute job enviado.")
        print("Execute Job ID:", matrix_job_ref.id)

    elif compile_status.status.value in terminal_errors:
        raise RuntimeError(f"El compile job terminó sin éxito: {compile_status}")
    else:
        print("La compilación continúa. Reejecuta esta celda más tarde.")


if matrix_job_ref is None:
    if matrix_compile_job_ref is None:
        print("No hay una matriz remota nueva que consultar.")
    else:
        print("Aún no existe execute job; espera a que termine la compilación.")

else:
    matrix_status = qnx.jobs.status(matrix_job_ref)
    print("Execute status:", matrix_status.status)
    print("Execute message:", matrix_status.message)

    queue_position = getattr(matrix_status, "queue_position", None)
    if queue_position is not None:
        print("Posición en cola:", queue_position)

    if matrix_status.status == qnx.jobs.JobStatusEnum.COMPLETED:
        matrix_result_refs, matrix_downloaded, matrix_counts, matrix_result_ids = (
            descargar_resultados_job(matrix_job_ref)
        )

        if len(matrix_downloaded) != len(matrix_pair_metadata):
            raise RuntimeError(
                "La cantidad de resultados no coincide con los pares enviados."
            )

        remote_matrix = np.eye(n_matrix_rows, dtype=float)
        if MATRIX_EXECUTE_DIAGONAL:
            remote_matrix.fill(0.0)
        remote_run_rows = []

        for metadata, downloaded_result, result_id in zip(
            matrix_pair_metadata,
            matrix_downloaded,
            matrix_result_ids,
        ):
            summary = resumen_kernel_desde_resultado(
                downloaded_result,
                matrix_input.shape[1],
            )
            i = metadata["matrix_i"]
            j = metadata["matrix_j"]
            remote_matrix[i, j] = summary["kernel_rate"]
            remote_matrix[j, i] = summary["kernel_rate"]

            remote_run_rows.append({
                **metadata,
                "result_id": result_id,
                "backend": MATRIX_BACKEND,
                "program_format": matrix_program_format,
                "n_qubits": matrix_input.shape[1],
                "zero_state": summary["zero_state"],
                "zero_count": summary["zero_count"],
                "shots": summary["shots"],
                "kernel_rate": summary["kernel_rate"],
            })

        matrix_result = {
            "kernel_matrix": remote_matrix,
            "run_summary": pd.DataFrame(remote_run_rows),
            "row_labels": MATRIX_ROWS,
            "n_circuits": len(matrix_pair_metadata),
            "n_shots_per_circuit": MATRIX_SHOTS,
        }
        K_selected = pd.DataFrame(
            remote_matrix,
            index=MATRIX_ROWS,
            columns=MATRIX_ROWS,
        )

        print("Matriz kernel reconstruida:")
        display(K_selected)
        print("Resumen compacto:")
        display(matrix_result["run_summary"])

        if SAVE_MATRIX_RUN:
            matrix_csv = guardar_matriz_kernel_run(
                matrix_result,
                source=f"nexus_{MATRIX_BACKEND}",
                run_id=str(matrix_job_ref.id),
                job_id=matrix_job_ref.id,
            )
            print("Run de matriz guardado en:", matrix_csv)

    elif matrix_status.status.value in terminal_errors:
        raise RuntimeError(f"El execute job terminó sin éxito: {matrix_status}")
    else:
        print("La ejecución continúa. Reejecuta esta celda más tarde.")

Execute status: JobStatusEnum.COMPLETED
Execute message: The job is completed.
Matriz kernel reconstruida:


,0,1,2,3
0,1.000,0.000,0.003,0.000
1,0.000,1.000,0.002,0.004
2,0.003,0.002,1.000,0.001
3,0.000,0.004,0.001,1.000


Resumen compacto:


,matrix_i,matrix_j,row_i,row_j,seed,result_id,backend,program_format,n_qubits,zero_state,zero_count,shots,kernel_rate
0,0,0,0,0,42,1e95fed7-79ec-4112-b7f0-cdfdd0f1ea80,H1-1LE,pytket_circuit,9,000000000,1000,1000,1.000
1,0,1,0,1,43,bcb5aba5-af5d-4363-ba94-8e17aafddd5b,H1-1LE,pytket_circuit,9,000000000,0,1000,0.000
2,0,2,0,2,44,e696c6aa-4c7a-4b6f-ac83-7b0ec8351db4,H1-1LE,pytket_circuit,9,000000000,3,1000,0.003
3,0,3,0,3,45,9a894030-fdd3-4cc3-9593-7d2183f66db1,H1-1LE,pytket_circuit,9,000000000,0,1000,0.000
4,1,1,1,1,47,5a785adb-1d09-4d38-9de8-db9d9d086153,H1-1LE,pytket_circuit,9,000000000,1000,1000,1.000
5,1,2,1,2,48,f9f537e5-3958-40e6-b310-30f71318df69,H1-1LE,pytket_circuit,9,000000000,2,1000,0.002
6,1,3,1,3,49,4586ca18-372c-4c01-8dfd-ed7d21797a5e,H1-1LE,pytket_circuit,9,000000000,4,1000,0.004
7,2,2,2,2,52,a7f1a89c-0d51-45c2-89a4-260088e0c959,H1-1LE,pytket_circuit,9,000000000,1000,1000,1.000
8,2,3,2,3,53,ab1f74a4-6820-4d2b-967d-6cc5c877e5cc,H1-1LE,pytket_circuit,9,000000000,1,1000,0.001
9,3,3,3,3,57,9b7a20b8-9ab8-4670-85f1-3626a5c1b9dc,H1-1LE,pytket_circuit,9,000000000,1000,1000,1.000


Run de matriz guardado en: data\runs\kernel_matrix_run_b3ab3978-87d6-4339-9c83-db8d081b6e13.csv
